# 第14章：知识检索（Knowledge Retrieval）

## 什么是 RAG？

**RAG（Retrieval Augmented Generation，检索增强生成）** 是一种将外部知识与 LLM 结合的技术。

### 为什么需要 RAG？

| 问题 | 说明 |
|------|------|
| 知识截止 | LLM 训练数据有截止日期，无法获取最新信息 |
| 幻觉问题 | LLM 可能编造不存在的事实 |
| 领域知识 | LLM 缺乏企业内部、专业领域的知识 |

### RAG 流程

```
用户提问 → 检索相关文档 → 将文档+问题传给LLM → 生成答案
```

### 三种检索方式

| 方式 | 原理 | 适用场景 |
|------|------|----------|
| 关键词匹配 | TF-IDF / BM25 | 简单问答，关键词明确 |
| 向量检索 | Embedding + 余弦相似度 | 语义理解，同义词多 |
| 混合检索 | 关键词 + 向量结合 | 复杂场景，精度要求高 |

本章使用 **TF-IDF 关键词匹配** 模拟 RAG 流程，专注学习流程设计。

In [ ]:
import os
import math
import re
from typing import List, Dict, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END


In [11]:

# 初始化 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0.3
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 步骤1：文档准备

创建一个中文知识库，包含3份公司制度文档。

In [12]:
# 创建知识库文档
knowledge_base = [
    Document(
        page_content="公司休假制度：员工入职满一年后享有5天年假，满五年后享有10天年假，满十年后享有15天年假。\n病假需提供医院证明，每月不超过3天。事假需提前3天申请，每月不超过2天。\n婚假为3天，产假按照国家规定执行。年假不可跨年累积，未休年假按日工资的300%补偿。",
        metadata={"source": "休假制度", "doc_id": "doc_01"}
    ),
    Document(
        page_content="公司报销流程：差旅费用需在出差结束后5个工作日内提交报销申请。\n住宿标准：一线城市不超过500元/晚，二线城市不超过300元/晚。\n餐饮补贴：每人每天100元。交通费用实报实销，需提供正规发票。\n单笔报销超过2000元需部门经理审批，超过5000元需财务总监审批。",
        metadata={"source": "报销流程", "doc_id": "doc_02"}
    ),
    Document(
        page_content="远程办公政策：员工每周可申请1-2天远程办公。\n需提前1天在OA系统提交申请，经直属主管审批。远程办公期间需保持在线，响应时间不超过30分钟。\n每月远程办公天数不超过8天。新入职员工试用期内不可申请远程办公。\n远程办公期间如遇重要会议，需到公司参加。",
        metadata={"source": "远程办公", "doc_id": "doc_03"}
    )
]

print(f"知识库文档数量: {len(knowledge_base)}")
for doc in knowledge_base:
    print(f"  - {doc.metadata['source']}: {len(doc.page_content)} 字")

知识库文档数量: 3
  - 休假制度: 126 字
  - 报销流程: 132 字
  - 远程办公: 125 字


## 步骤2：文档分块（Text Splitting）

将长文档切分为较小的块（chunks），便于精确检索。

分块策略：
- `chunk_size`：每块最大字符数
- `chunk_overlap`：相邻块重叠字符数，避免语义断裂

In [13]:
def split_text(text: str, chunk_size: int = 200, chunk_overlap: int = 50) -> List[str]:
    """简单的文本分块函数"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - chunk_overlap
    return chunks

# 对知识库文档进行分块
all_chunks = []
for doc in knowledge_base:
    chunks = split_text(doc.page_content, chunk_size=150, chunk_overlap=30)
    for i, chunk in enumerate(chunks):
        all_chunks.append(Document(
            page_content=chunk,
            metadata={**doc.metadata, "chunk_index": i}
        ))

print(f"分块后文档数量: {len(all_chunks)}")
for i, chunk in enumerate(all_chunks):
    print(f"  块{i+1} [{chunk.metadata['source']}]: {chunk.page_content[:50]}...")

分块后文档数量: 6
  块1 [休假制度]: 公司休假制度：员工入职满一年后享有5天年假，满五年后享有10天年假，满十年后享有15天年假。
病假需...
  块2 [休假制度]: 00%补偿。...
  块3 [报销流程]: 公司报销流程：差旅费用需在出差结束后5个工作日内提交报销申请。
住宿标准：一线城市不超过500元/晚...
  块4 [报销流程]: 000元需财务总监审批。...
  块5 [远程办公]: 远程办公政策：员工每周可申请1-2天远程办公。
需提前1天在OA系统提交申请，经直属主管审批。远程办...
  块6 [远程办公]: 公司参加。...


## 步骤3：检索机制（TF-IDF 模拟）

使用 TF-IDF 算法计算查询与文档的相似度，返回最相关的文档块。

### TF-IDF 原理

- **TF（词频）**：词在文档中出现的频率
- **IDF（逆文档频率）**：词的稀有程度，越稀有权重越高
- **TF-IDF = TF × IDF**：综合衡量词的重要性

In [14]:
def tokenize(text: str) -> List[str]:
    """中文分词（简单按字符和标点分割）"""
    # 去除标点，按字符分割
    text = re.sub(r'[\\s\\W]+', '', text)
    return list(text)

def compute_tf(tokens: List[str]) -> Dict[str, float]:
    """计算词频 TF"""
    tf = {}
    for token in tokens:
        tf[token] = tf.get(token, 0) + 1
    total = len(tokens)
    return {k: v / total for k, v in tf.items()}

def compute_idf(documents: List[List[str]]) -> Dict[str, float]:
    """计算逆文档频率 IDF"""
    idf = {}
    total_docs = len(documents)
    all_tokens = set()
    for doc in documents:
        all_tokens.update(doc)
    
    for token in all_tokens:
        doc_count = sum(1 for doc in documents if token in doc)
        idf[token] = math.log(total_docs / (1 + doc_count))
    return idf

def tfidf_similarity(query_tokens: List[str], doc_tokens: List[str], 
                     idf: Dict[str, float]) -> float:
    """计算 TF-IDF 相似度"""
    query_tf = compute_tf(query_tokens)
    doc_tf = compute_tf(doc_tokens)
    
    score = 0.0
    for token in query_tokens:
        if token in doc_tf and token in idf:
            score += query_tf.get(token, 0) * doc_tf[token] * idf[token]
    return score

# 构建索引
chunk_tokens = [tokenize(chunk.page_content) for chunk in all_chunks]
idf = compute_idf(chunk_tokens)

print("TF-IDF 索引构建完成！")
print(f"词汇表大小: {len(idf)} 个字符")

TF-IDF 索引构建完成！
词汇表大小: 151 个字符


In [15]:
def retrieve(query: str, top_k: int = 2) -> List[Document]:
    """检索最相关的文档块"""
    query_tokens = tokenize(query)
    scores = []
    
    for i, doc_tokens in enumerate(chunk_tokens):
        score = tfidf_similarity(query_tokens, doc_tokens, idf)
        scores.append((score, i))
    
    # 按分数降序排列
    scores.sort(reverse=True)
    results = []
    for score, idx in scores[:top_k]:
        if score > 0:
            results.append(all_chunks[idx])
    return results

# 测试检索
test_query = "年假有几天"
results = retrieve(test_query)
print(f"查询: {test_query}")
print(f"检索到 {len(results)} 个相关文档:")
for i, doc in enumerate(results):
    print(f"  {i+1}. [{doc.metadata['source']}] {doc.page_content[:80]}...")

查询: 年假有几天
检索到 2 个相关文档:
  1. [休假制度] 公司休假制度：员工入职满一年后享有5天年假，满五年后享有10天年假，满十年后享有15天年假。
病假需提供医院证明，每月不超过3天。事假需提前3天申请，每月不超过...
  2. [远程办公] 远程办公政策：员工每周可申请1-2天远程办公。
需提前1天在OA系统提交申请，经直属主管审批。远程办公期间需保持在线，响应时间不超过30分钟。
每月远程办公天数...


## 步骤4：RAG 生成

将检索到的文档作为上下文，结合用户问题，让 LLM 生成答案。

### Prompt 设计

```
你是一个问答助手。根据以下参考文档回答用户问题。
如果文档中没有相关信息，请如实说明。

参考文档：{context}
用户问题：{question}
```

In [16]:
# RAG 问答函数
rag_prompt = ChatPromptTemplate.from_template(
    "你是一个公司制度问答助手。根据以下参考文档回答用户问题。\n"
    "如果文档中没有相关信息，请如实说明'不知道'。\n"
    "请用中文回答，保持简洁。\n\n"
    "参考文档：\n{context}\n\n"
    "用户问题：{question}\n\n"
    "回答："
)

rag_chain = rag_prompt | llm | StrOutputParser()

def ask_with_rag(question: str) -> str:
    """带 RAG 的问答"""
    # 1. 检索相关文档
    docs = retrieve(question, top_k=2)
    
    # 2. 拼接上下文
    context = "\n".join([doc.page_content for doc in docs])
    
    # 3. 生成答案
    answer = rag_chain.invoke({"context": context, "question": question})
    return answer, docs

# 测试 RAG 问答
print("=== RAG 问答测试 ===")
print()

questions = [
    "年假有几天？",
    "差旅报销标准是什么？",
    "如何申请远程办公？"
]

for q in questions:
    print(f"问题: {q}")
    answer, docs = ask_with_rag(q)
    print(f"检索来源: {[d.metadata['source'] for d in docs]}")
    print(f"回答: {answer}")
    print("-" * 50)

=== RAG 问答测试 ===

问题: 年假有几天？
检索来源: ['休假制度', '远程办公']
回答: 根据公司制度，年假天数根据入职年限确定：
- 入职满1年：5天
- 满5年：10天
- 满10年：15天
--------------------------------------------------
问题: 差旅报销标准是什么？
检索来源: ['报销流程']
回答: 根据公司制度，差旅报销标准如下：

1. **住宿标准**  
   - 一线城市：不超过500元/晚  
   - 二线城市：不超过300元/晚  

2. **餐饮补贴**  
   - 每人每天100元（固定补贴）  

3. **交通费用**  
   - 实报实销，需提供正规发票  

4. **审批要求**  
   - 单笔报销超过2000元需部门经理审批  
   - 超过5000元需财务总监审批  

5. **提交时限**  
   - 出差结束后5个工作日内提交报销申请  

（注：其他未提及的费用标准，文档中未说明。）
--------------------------------------------------
问题: 如何申请远程办公？
检索来源: ['远程办公', '远程办公']
回答: 根据文档，申请远程办公需提前1天在OA系统提交申请，经直属主管审批。同时需注意：每周可申请1-2天，每月不超过8天；试用期员工不可申请；如遇重要会议需到公司参加。
--------------------------------------------------


## 步骤5：LangGraph RAG 流程

使用 LangGraph 将 RAG 流程编排为图：

```
retrieve → generate → END
```

In [17]:
# LangGraph RAG 流程
class RAGState(TypedDict):
    question: str
    documents: List[Document]
    generation: str

def retrieve_node(state: RAGState) -> RAGState:
    """检索节点：根据问题检索相关文档"""
    question = state["question"]
    documents = retrieve(question, top_k=2)
    return {"question": question, "documents": documents, "generation": ""}

def generate_node(state: RAGState) -> RAGState:
    """生成节点：根据检索结果生成答案"""
    question = state["question"]
    documents = state["documents"]
    
    context = "\n".join([doc.page_content for doc in documents])
    generation = rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "documents": documents, "generation": generation}

# 构建图
workflow = StateGraph(RAGState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)
app = workflow.compile()

print("LangGraph RAG 流程构建完成！")

LangGraph RAG 流程构建完成！


In [18]:
# 运行 LangGraph RAG
print("=== LangGraph RAG 测试 ===")
print()

test_questions = [
    "年假有几天？",
    "差旅报销标准是什么？",
    "如何申请远程办公？",
    "加班费怎么算？"  # 知识库中没有的信息
]

for q in test_questions:
    print(f"问题: {q}")
    result = app.invoke({"question": q, "documents": [], "generation": ""})
    print(f"检索来源: {[d.metadata['source'] for d in result['documents']]}")
    print(f"回答: {result['generation']}")
    print("=" * 50)
    print()

=== LangGraph RAG 测试 ===

问题: 年假有几天？
检索来源: ['休假制度', '远程办公']
回答: 根据公司制度，年假天数如下：
- 入职满1年：5天
- 满5年：10天
- 满10年：15天

年假不可跨年累积，未休部分按日工资的300%补偿。

问题: 差旅报销标准是什么？
检索来源: ['报销流程']
回答: 根据公司制度，差旅报销标准如下：
1. **住宿标准**：一线城市不超过500元/晚，二线城市不超过300元/晚。
2. **餐饮补贴**：每人每天100元。
3. **交通费用**：实报实销，需提供正规发票。

问题: 如何申请远程办公？
检索来源: ['远程办公', '远程办公']
回答: 根据公司政策，申请远程办公需提前1天在OA系统提交申请，并经直属主管审批。

问题: 加班费怎么算？
检索来源: ['远程办公', '报销流程']
回答: 不知道。参考文档中没有关于加班费计算的相关信息。



## 总结

### RAG 流程

```
文档加载 → 分块 → 索引 → 检索 → 生成
```

### RAG 优缺点

| 优点 | 缺点 |
|------|------|
| 减少幻觉 | 检索质量影响答案质量 |
| 知识可更新 | 需要维护知识库 |
| 可追溯来源 | 增加延迟和成本 |
| 保护数据隐私 | 分块策略影响效果 |

### 检索方式对比

| 方式 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| 关键词匹配 | TF-IDF / BM25 | 简单快速 | 无法理解语义 |
| 向量检索 | Embedding + 相似度 | 语义理解 | 需要 Embedding 模型 |
| 混合检索 | 两者结合 | 效果最好 | 复杂度高 |

### 应用场景

| 场景 | 说明 |
|------|------|
| 企业知识库 | 内部制度、流程、FAQ |
| 客服系统 | 产品手册、常见问题 |
| 法律咨询 | 法规、案例检索 |
| 医疗问答 | 医学文献、药品说明 |
| 技术文档 | API 文档、开发指南 |